**Data Loading and Preparation**

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split

#setup Context
catalog = "dev"
schema = "ecommerce_governed"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

#1. Load Data from Gold Layer
df_gold = spark.table("gold_events").toPandas()

#2. Feature Selection
df_clean = df_gold.fillna(0)

x = df_clean[["unique_views"]]
y = df_clean["unique_purchases"]

#Train/Test Split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2 , random_state=42)

display(x_train.head(5))
display(x_test.head(5))


**Experiment 1 - Linear Regression**

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

user = 'rainbow.mem5@gmail.com'
mlflow.set_experiment(f"/Users/{user}/Day12_ML")

with mlflow.start_run(run_name = "Linear Regression"):
  #Create Model
  model_type = "Linear Regression"
  mlflow.log_param("model_type", model_type)
  mlflow.log_param("features", "unique_views")

  #Train Model
  lr = LinearRegression()
  lr.fit(x_train, y_train)

  # 3. Predict and Evaluate
  predictions = lr.predict(x_test)
  r2 = r2_score(y_test, predictions)
  mse = mean_squared_error(y_test, predictions)
  mae = mean_absolute_error(y_test, predictions)

#4. Log Metrics Performance
mlflow.log_metric("r2_score", r2)
mlflow.log_metric("mse", mse)
mlflow.log_metric("mae", mae)

#5. Log Model
mlflow.sklearn.log_model(lr, "model")